In [ ]:
!pip install chromadb sentence-transformers rank-bm25 pymupdf langchain \
             langchain-community google-generativeai gradio \
             python-dotenv tqdm -q
!pip install datasets -q

In [1]:
import os
import re
import json
import pickle
import hashlib
from pathlib import Path
from datetime import datetime

import fitz  # PyMuPDF
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient

print('import complete')

import complete


In [16]:
# ── Paths ──────────────────────────────────────────────────────────
BASE_DIR     = Path("/kaggle/working")
DATA_DIR     = Path("/kaggle/input")          # adjust after you attach dataset
CHROMA_DIR   = BASE_DIR / "chroma_db"
BM25_PATH    = BASE_DIR / "bm25_index.pkl"
CHUNKS_PATH  = BASE_DIR / "all_chunks.pkl"   # save raw chunks too, useful later

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# ── Chunking config ─────────────────────────────────────────────────
CHUNK_SIZE    = 512   # tokens (approx chars/4)
CHUNK_OVERLAP = 50
CHAR_LIMIT    = CHUNK_SIZE * 4        # rough char approximation
OVERLAP_CHARS = CHUNK_OVERLAP * 4

# ── Embedding model ─────────────────────────────────────────────────
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
BATCH_SIZE    = 256   # safe for T4 x2
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_2 = user_secrets.get_secret("KAGGLE_USERNAME") 

print("✓ config ready")

✓ config ready


In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"GPUs available: {torch.cuda.device_count()}")

# 2. Pass the token parameter to the SentenceTransformer
embedder = SentenceTransformer(
    EMBED_MODEL, 
    device=device, 
    token=HF_TOKEN # <-- Added token here
)

# quick sanity check
test_emb = embedder.encode(["test legal clause"], normalize_embeddings=True)
print(f"✓ embedding model loaded | dim={test_emb.shape[1]}")

Using device: cuda
GPUs available: 2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ embedding model loaded | dim=384


In [5]:
def clean_contract_text(text: str) -> str:
    """
    Clean raw contract text extracted from TXT or PDF.
    Removes page headers, footers, excessive whitespace,
    page numbers, and other noise while preserving structure.
    """
    # remove page number lines (standalone numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # remove common header/footer patterns
    text = re.sub(r'^\s*(page\s+\d+(?:\s+of\s+\d+)?)\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*(confidential|draft|execution copy|exhibit\s+[a-z])\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)

    # collapse 3+ newlines to 2 (preserve section breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # fix mid-line hyphenation (PDF artifacts like "indem-\nnification")
    text = re.sub(r'-\n([a-z])', r'\1', text)

    # remove null bytes and weird unicode control chars
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # collapse multiple spaces (but not newlines)
    text = re.sub(r'[^\S\n]+', ' ', text)

    return text.strip()


# test it
sample = "Page 1\n\n\n\nThis is a clause.\n\n\n\nPage 2\n\n"
print(repr(clean_contract_text(sample)))
print("✓ cleaner ready")

'This is a clause.'
✓ cleaner ready


In [28]:
# The official 41 CUAD (Contract Understanding Atticus Dataset) categories
CLAUSE_PATTERNS = {
    # General Contract Details
    "document_name": r'\b(agreement|contract|amendment|addendum|statement\s+of\s+work)\b',
    "parties": r'\b(entered\s+into|by\s+and\s+between|by\s+and\s+among)\b',
    "agreement_date": r'\b(dated\s+as\s+of|agreement\s+date)\b',
    "effective_date": r'\b(effective\s+date|shall\s+become\s+effective)\b',
    "expiration_date": r'\b(expiration\s+date|expire(s)?\s+on)\b',
    "renewal_term": r'\brenewal\s+term\b|\bauto(matic)?\s*renew',
    "notice_period_to_terminate_renewal": r'\bnotice\s+(to\s+terminate|of\s+non[-\s]renewal)\b',
    "governing_law": r'\bgoverning\s+law\b|\bchoice\s+of\s+law\b',
    
    # Financial & Commercial Terms
    "most_favored_nation": r'\b(most\s+favored\s+nation|mfn)\b',
    "revenue_profit_sharing": r'\b(revenue|profit)\s+shar(e|ing)\b',
    "price_restrictions": r'\b(price\s+(restriction|protection|decrease)|lock[-\s]in\s+price)\b',
    "minimum_commitment": r'\bminimum\s+(purchase|commit|order|volume)\b',
    "volume_restriction": r'\b(volume\s+restriction|maximum\s+quantity)\b',
    
    # Restrictive Covenants
    "non_compete": r'\bnon[-\s]compete\b|\bnot\s+to\s+compete\b',
    "exclusivity": r'\b(exclusive|exclusivity)\b',
    "no_solicit_of_customers": r'\b(non[-\s]solicit|not\s+to\s+solicit)\s+(customers|clients)\b',
    "competitive_restriction_exception": r'\bexception(s)?\s+to\s+(non[-\s]compete|exclusivity)\b',
    "no_solicit_of_employees": r'\b(non[-\s]solicit|not\s+to\s+solicit)\s+(employees|personnel)\b',
    "non_disparagement": r'\b(non[-\s]disparage|disparagement)\b',
    
    # Termination & Assignment
    "termination_for_convenience": r'\btermination\s+for\s+convenience\b',
    "rofr_rofo_rofn": r'\bright\s+of\s+first\s+(refusal|offer|negotiation)|rofr|rofo|rofn\b',
    "change_of_control": r'\bchange\s+of\s+control\b',
    "anti_assignment": r'\b(anti[-\s]assignment|assignment\s+and\s+delegation)\b|\bassign\s+this\s+agreement\b',
    "post_termination_services": r'\b(post[-\s]termination\s+services|transition\s+assistance)\b',
    
    # Intellectual Property & Licensing
    "ip_ownership_assignment": r'\bintellectual\s+property\s+ownership\b|\bip\s+ownership\b|\bassignment\s+of\s+(ip|intellectual)\b',
    "joint_ip_ownership": r'\b(joint|shared)\s+(intellectual\s+property|ip|ownership)\b',
    "license_grant": r'\b(grant\s+of\s+license|license\s+grant)\b',
    "non_transferable_license": r'\bnon[-\s]transferable\s+license\b',
    "affiliate_license_licensor": r'\blicensor\s+affiliate(s)?\b',
    "affiliate_license_licensee": r'\blicensee\s+affiliate(s)?\b',
    "unlimited_license": r'\b(unlimited|all[-\s]you[-\s]can[-\s]eat)\s+license\b',
    "perpetual_license": r'\b(irrevocable|perpetual)\s+license\b',
    "source_code_escrow": r'\b(source\s+code\s+escrow|escrow\s+agent)\b',
    
    # Liability, Risk & Audits
    "audit_rights": r'\baudit\s+right(s)?\b',
    "uncapped_liability": r'\b(uncapped|unlimited)\s+liability\b',
    "cap_on_liability": r'\blimitation\s+of\s+liability\b|\bcap\s+on\s+liability\b',
    "liquidated_damages": r'\bliquidated\s+damages\b',
    "warranty_duration": r'\bwarranty\s+(duration|period)\b',
    "insurance": r'\binsurance\s+(policy|coverage|requirements)\b',
    "covenant_not_to_sue": r'\bcovenant\s+not\s+to\s+sue\b',
    "third_party_beneficiary": r'\bthird[-\s]party\s+beneficiar(y|ies)\b'
}

SECTION_HEADER_RE = re.compile(
    r'(?:^|\n)('
    r'(?:\d+\.)+\d*\s+[A-Z]'          # 1.1 HEADING
    r'|[A-Z][A-Z\s]{4,}(?:\.|:|\n)'   # ALL CAPS HEADING
    r'|Section\s+\d+'                 # Section 12
    r'|Article\s+\d+'                 # Article 5
    r'|ARTICLE\s+[IVXLC]+'            # ARTICLE VI
    r')',
    re.MULTILINE
)

def detect_clause_type(text: str) -> str:
    text_lower = text.lower()
    for clause_type, pattern in CLAUSE_PATTERNS.items():
        # skip document_name in first pass — too generic
        if clause_type == "document_name":
            continue
        if re.search(pattern, text_lower):
            return clause_type
    # only fall back to document_name if nothing else matched
    if re.search(CLAUSE_PATTERNS["document_name"], text_lower):
        return "document_name"
    return "unknown"

def chunk_contract(text: str, contract_id: str,
                   contract_name: str, source: str = "cuad") -> list[dict]:
    """
    Split a cleaned contract into clause-aware chunks with metadata.
    Splits on structural signals first, keeping headers attached to content, 
    then enforces size limits.
    """
    chunks = []
    chunk_idx = 0

    # split on section headers 
    sections = SECTION_HEADER_RE.split(text)
    pieces = []
    
    # The first item is any introductory text before the first header
    if sections[0].strip():
        pieces.append(sections[0].strip())
        
    # re.split with a capture group returns alternating [header, content, header, content...]
    # We step through by 2s to stitch the header back together with its content
    for i in range(1, len(sections), 2):
        header = sections[i].strip()
        content = sections[i+1].strip() if i+1 < len(sections) else ""
        combined_piece = f"{header}\n{content}".strip()
        if combined_piece:
            pieces.append(combined_piece)

    # now enforce size limit within each piece
    for piece in pieces:
        if len(piece) <= CHAR_LIMIT:
            if len(piece) > 80:  # skip tiny fragments
                chunks.append(piece)
        else:
            # piece too long — sliding window split
            start = 0
            while start < len(piece):
                end = start + CHAR_LIMIT
                chunk_text = piece[start:end]
                if len(chunk_text) > 80:
                    chunks.append(chunk_text)
                # Ensure we make forward progress
                start = end - OVERLAP_CHARS if end < len(piece) else len(piece)

    # build metadata for each chunk
    result = []
    for raw_chunk in chunks:
        clause_type = detect_clause_type(raw_chunk)
        result.append({
            "text": raw_chunk,
            "metadata": {
                "contract_id":    contract_id,
                "contract_name":  contract_name,
                "source":         source,
                "chunk_index":    chunk_idx,
                "clause_type":    clause_type,
                "auto_tagged":    False,
                "char_length":    len(raw_chunk),
            }
        })
        chunk_idx += 1

    return result

print("✓ chunker ready")

✓ chunker ready


In [17]:
 !kaggle datasets download -d konradb/atticus-open-contract-dataset-aok-beta \
    --path /kaggle/working/cuad_data --unzip

Dataset URL: https://www.kaggle.com/datasets/konradb/atticus-open-contract-dataset-aok-beta
License(s): Attribution 4.0 International (CC BY 4.0)
100%|█████████████████████████████████████████| 102M/102M [00:00<00:00, 126MB/s]



In [29]:
TXT_DIR = Path("/kaggle/working/cuad_data/CUAD_v1/full_contract_txt")

txt_files = sorted(TXT_DIR.glob("*.txt"))
print(f"Found {len(txt_files)} contracts")

all_chunks = []

for txt_path in tqdm(txt_files, desc="Chunking contracts"):
    # skip readme
    if "readme" in txt_path.name.lower():
        continue
        
    raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
    cleaned  = clean_contract_text(raw_text)

    contract_name = txt_path.stem
    contract_id   = hashlib.md5(contract_name.encode()).hexdigest()[:12]

    chunks = chunk_contract(
        text          = cleaned,
        contract_id   = contract_id,
        contract_name = contract_name,
        source        = "cuad"
    )
    all_chunks.extend(chunks)

print(f"\n✓ total chunks: {len(all_chunks)}")
print(f"  avg chars/chunk: {sum(ch['metadata']['char_length'] for ch in all_chunks) // len(all_chunks)}")

with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(all_chunks, f)

print(f"✓ chunks saved to {CHUNKS_PATH}")

Found 510 contracts


Chunking contracts: 100%|██████████| 510/510 [00:16<00:00, 30.36it/s]



✓ total chunks: 21890
  avg chars/chunk: 1255
✓ chunks saved to /kaggle/working/all_chunks.pkl


In [30]:
print("Building BM25 index...")

# tokenize — simple whitespace + lowercase for legal text
def tokenize(text: str) -> list[str]:
    return re.findall(r'\b[a-z]{2,}\b', text.lower())

corpus_tokens = [tokenize(chunk["text"]) for chunk in tqdm(all_chunks, desc="Tokenizing")]
bm25 = BM25Okapi(corpus_tokens)

with open(BM25_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "corpus_tokens": corpus_tokens}, f)

print(f"✓ BM25 index saved | vocab size ≈ {len(set(t for doc in corpus_tokens for t in doc))}")

Building BM25 index...


Tokenizing: 100%|██████████| 21890/21890 [00:01<00:00, 18013.17it/s]


✓ BM25 index saved | vocab size ≈ 30273


In [31]:
# init ChromaDB
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False)
)

# drop and recreate if re-running
try:
    client.delete_collection("contracts")
except:
    pass

collection = client.create_collection(
    name="contracts",
    metadata={"hnsw:space": "cosine"}
)

print(f"✓ ChromaDB collection created at {CHROMA_DIR}")
print(f"  embedding {len(all_chunks)} chunks in batches of {BATCH_SIZE}...")

# embed + insert in batches
for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Embedding"):
    batch = all_chunks[i : i + BATCH_SIZE]

    texts     = [c["text"]     for c in batch]
    metadatas = [c["metadata"] for c in batch]
    ids       = [f"{m['contract_id']}_{m['chunk_index']}" for m in metadatas]

    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        device=device
    ).tolist()

    collection.add(
        ids        = ids,
        embeddings = embeddings,
        documents  = texts,
        metadatas  = metadatas
    )

print(f"\n✓ ChromaDB populated | total docs: {collection.count()}")

✓ ChromaDB collection created at /kaggle/working/chroma_db
  embedding 21890 chunks in batches of 256...


Embedding: 100%|██████████| 86/86 [03:18<00:00,  2.30s/it]


✓ ChromaDB populated | total docs: 21890


In [32]:
# test dense retrieval
query = "what is the governing law of this agreement"
q_emb = embedder.encode([query], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings = q_emb,
    n_results        = 5,
    include          = ["documents", "metadatas", "distances"]
)

print("=== Top 5 dense results ===\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"[{i+1}] contract={meta['contract_name']} | clause={meta['clause_type']} | score={1-dist:.3f}")
    print(f"     {doc[:120]}...\n")

=== Top 5 dense results ===

[1] contract=NEONSYSTEMSINC_03_01_1999-EX-10.5-DISTRIBUTOR AGREEMENT_New | clause=governing_law | score=0.845
     Section 15
.6 GOVERNING LAW. THIS AGREEMENT SHALL BE GOVERNED BY AND CONSTRUED IN ACCORDANCE WITH THE LAWS OF THE SIATE ...

[2] contract=ENTRUSTINC_07_24_1998-EX-10.5-STRATEGIC ALLIANCE AGREEMENT | clause=governing_law | score=0.825
     Section 12
.06. Governing Law. This Agreement shall be governed by and be --------- ---- construed in accordance with th...

[3] contract=Reinsurance Group of America, Incorporated - A_R REMARKETING  AGREEMENT | clause=governing_law | score=0.821
     Section 16
. Governing Law. This Agreement shall be governed by, and construed in accordance with, the laws of the State...

[4] contract=WestPharmaceuticalServicesInc_20200116_8-K_EX-10.1_11947529_EX-10.1_Supply Agreement | clause=governing_law | score=0.817
     GOVERNING LAW
This Agreement shall be governed and construed in accordance with the law set forth in

In [33]:
import shutil

print("Zipping artifacts...")
shutil.make_archive(str(BASE_DIR / "chroma_db"), 'zip', str(CHROMA_DIR))
print(f"✓ chroma_db.zip created")

# list what to download
for f in BASE_DIR.glob("*.pkl"):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {size_mb:.1f} MB")

zip_path = BASE_DIR / "chroma_db.zip"
size_mb = zip_path.stat().st_size / 1e6
print(f"  chroma_db.zip: {size_mb:.1f} MB")

print("\n✓ Download these 3 files from Kaggle output:")
print("  1. chroma_db.zip")
print("  2. bm25_index.pkl")
print("  3. all_chunks.pkl")

Zipping artifacts...
✓ chroma_db.zip created
  all_chunks.pkl: 28.7 MB
  bm25_index.pkl: 48.0 MB
  chroma_db.zip: 229.3 MB

✓ Download these 3 files from Kaggle output:
  1. chroma_db.zip
  2. bm25_index.pkl
  3. all_chunks.pkl
